In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

dataset = dataset.map(
    lambda example: {
        "combined_text": example["prompt"] + " " + example["A"]
    }
)

print(len(dataset[51]["combined_text"]))

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sep_token_id = tokenizer.sep_token_id

print(sep_token_id)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

encoded = tokenizer(
    list(dataset["prompt"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(dataset[0]["prompt"], return_tensors="pt")

outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

In [ ]:
cls_embedding = outputs.last_hidden_state[0, 0]

answer = cls_embedding[:5].sum().item()

print(round(answer, 4))

In [ ]:
from transformers import AutoModel
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)
text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

fusion_index = tokens.index("fusion")
print("fusion index:", fusion_index)

attention = outputs.attentions[-1][0, 0]

weight = attention[0, fusion_index].item()

print(f"{weight:.4f}")

In [ ]:
from sentence_transformers import SentenceTransformer, util
import traceback

try:
    print("Loading model...")
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    print("Model loaded")

    print("Encoding prompt...")
    prompt_embedding = model.encode(dataset[0]["prompt"], convert_to_tensor=True)
    print("Prompt encoded")

    print("Encoding option...")
    option_embedding = model.encode(dataset[0]["B"], convert_to_tensor=True)
    print("Option encoded")

    print("Computing similarity...")
    similarity = util.cos_sim(prompt_embedding, option_embedding)

    print(similarity.item())

except Exception:
    traceback.print_exc()

In [ ]:
import torch
import sentence_transformers
import transformers
from sentence_transformers import SentenceTransformer, util

print(sentence_transformers.__version__)
print(transformers.__version__)
print(torch.__version__)
print(torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(f"{similarity.item():.4f}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import numpy as np
import torch

vectorizer = TfidfVectorizer()

tfidf_top3 = []

for row in dataset:

    texts = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    tfidf = vectorizer.fit_transform(texts)

    similarities = cosine_similarity(tfidf[0], tfidf[1:]).flatten()

    ranking = np.argsort(similarities)[::-1]

    options = np.array(["A","B","C","D","E"])

    tfidf_top3.append(options[ranking][:3].tolist())



model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

minilm_top3 = []

for row in dataset:

    texts = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    embeddings = model.encode(
        texts,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    similarities = util.cos_sim(
        embeddings[0],
        embeddings[1:]
    ).cpu().numpy().flatten()

    ranking = np.argsort(similarities)[::-1]

    options = np.array(["A","B","C","D","E"])

    minilm_top3.append(options[ranking][:3].tolist())


def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    score = 0.0
    hits = 0

    for i,p in enumerate(predicted):
        if p == actual:
            hits += 1
            score += hits/(i+1)

    return score

map3 = np.mean([
    apk(row["answer"], pred)
    for row,pred in zip(dataset,minilm_top3)
])

print("MiniLM MAP@3 =", round(map3,4))


count = 0

for row,tfidf_pred,minilm_pred in zip(
    dataset,
    tfidf_top3,
    minilm_top3
):

    answer = row["answer"]

    if answer not in tfidf_pred and answer in minilm_pred:
        count += 1

print("Improvement Count =", count)

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

sequence = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

result = classifier(sequence, candidate_labels)

print(result["labels"][0])
print(f"{result['scores'][0]:.4f}")

In [ ]:
result_multi = classifier(
    sequence,
    candidate_labels,
    multi_label=True
)

softmax_sum = sum(result["scores"])

sigmoid_sum = sum(result_multi["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax Sum:", softmax_sum)
print("Sigmoid Sum:", sigmoid_sum)
print(f"{difference:.4f}")

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    device=0 if torch.cuda.is_available() else -1
)

text = (
    f"Question: {dataset[0]['prompt']}. "
    f"Is the correct answer A: {dataset[0]['A']} "
    f"or B: {dataset[0]['B']}? "
    f"Answer with just the letter A or B."
)

result = generator(
    text,
    max_new_tokens=5
)

print(result[0]["generated_text"])